# Keras Notes

Based on the `keras/` directory in this repository.

## Importing Keras: Old vs New

### Old Keras (standalone, pre-TensorFlow 2.0)
```python
import keras
```
Keras used to be a standalone, high-level API that could run on top of TensorFlow, Theano, or CNTK as backends.

### New Keras (TensorFlow 2.x)
```python
import tensorflow.keras as K
```
Since TensorFlow 2.0 (2019), Keras is the **official high-level API** bundled inside TensorFlow as `tf.keras`. The standalone `keras` package is still available but not recommended — you should use `tf.keras`.

### Key Difference
- **Old:** Standalone library (`pip install keras`), pluggable backends (TensorFlow, Theano, CNTK). Last standalone version was Keras 2.x.
- **New:** `tf.keras` — TensorFlow's own implementation of the Keras API spec. Fully integrated with TF features (eager execution, `tf.data`, SavedModel format). No backend switching.
- **Relationship:** `tf.keras` is the **canonical** Keras now. The standalone Keras library is maintained but `tf.keras` is what you should use when working with TensorFlow.

> **This project uses:** `import tensorflow.keras as K`

## 1. Building a Sequential Model

**File:** `0-sequential.py`

The `Sequential` API is a linear stack of layers. You add layers one by one.

In [ ]:
import tensorflow.keras as K

def build_model(nx, layers, activations, lambtha, keep_prob):
    model = K.Sequential()
    for i in range(len(layers)):
        if i == 0:
            model.add(
                K.layers.Dense(
                    layers[i],
                    activation=activations[i],
                    kernel_regularizer=K.regularizers.l2(lambtha),
                    input_shape=(nx,)
                )
            )
        else:
            model.add(
                K.layers.Dense(
                    layers[i],
                    activation=activations[i],
                    kernel_regularizer=K.regularizers.l2(lambtha)
                )
            )
        if i != len(layers) - 1:
            model.add(K.layers.Dropout(1 - keep_prob))
    return model

**Key points:**
- `K.Sequential()` — linear stack of layers
- `model.add(layer)` — append layers
- `K.layers.Dense(...)` — fully connected layer with L2 regularization
- `K.layers.Dropout(rate)` — dropout for regularization
- `input_shape=(nx,)` — specify input dimensions on the first layer only

## 2. Building a Model with the Functional API

**File:** `1-input.py`

The Functional API is more flexible — you define an input tensor and pass it through layers explicitly.

In [ ]:
def build_model_functional(nx, layers, activations, lambtha, keep_prob):
    inputs = K.Input(shape=(nx,))
    a = inputs
    for i in range(len(layers)):
        a = K.layers.Dense(
            layers[i],
            activation=activations[i],
            kernel_regularizer=K.regularizers.l2(lambtha)
        )(a)
        if i != len(layers) - 1:
            a = K.layers.Dropout(1 - keep_prob)(a)
    model = K.Model(inputs=inputs, outputs=a)
    return model

**Key points:**
- `K.Input(shape=(nx,))` — define the input tensor
- Layers are called as functions: `K.layers.Dense(...)(a)`
- `K.Model(inputs=..., outputs=...)` — wrap into a model
- Functional API supports branching, multiple inputs/outputs, and non-sequential topologies

## 3. Compiling a Model (Optimizer, Loss, Metrics)

**File:** `2-optimize.py`

In [ ]:
def optimize_model(network, alpha, beta1, beta2):
    network.compile(
        loss='categorical_crossentropy',
        optimizer=K.optimizers.Adam(learning_rate=alpha,
                                    beta_1=beta1,
                                    beta_2=beta2),
        metrics=['accuracy']
    )

**Key points:**
- `model.compile(loss, optimizer, metrics)` — configure the model for training
- `K.optimizers.Adam(...)` — Adam optimizer with configurable betas
- Loss: `'categorical_crossentropy'` for multi-class classification
- Metrics: list of strings like `['accuracy']`

## 4. One-Hot Encoding

**File:** `3-one_hot.py`

In [ ]:
def one_hot(labels, classes=None):
    return K.utils.to_categorical(labels, num_classes=classes)

**Key points:**
- `K.utils.to_categorical(labels, num_classes)` — convert integer labels to one-hot vectors

## 5. Training Basics

**File:** `4-train.py`

In [ ]:
def train_model(network, data, labels, batch_size, epochs,
                verbose=True, shuffle=False):
    history = network.fit(
        data,
        labels,
        batch_size=batch_size,
        epochs=epochs,
        verbose=verbose,
        shuffle=shuffle,
    )
    return history

**Key points:**
- `model.fit(x, y, batch_size, epochs, ...)` — train the model
- Returns a `History` object with training metrics per epoch
- `verbose`: 0 (silent), 1 (progress bar), 2 (one line per epoch)
- `shuffle`: whether to shuffle training data before each epoch

## 6. Training with Validation Data

**File:** `5-train.py`

In [ ]:
def train_model_val(network, data, labels, batch_size, epochs,
                    validation_data=None, verbose=True, shuffle=False):
    history = network.fit(
        data, labels,
        batch_size=batch_size, epochs=epochs,
        verbose=verbose, shuffle=shuffle,
        validation_data=validation_data
    )
    return history

**Key points:**
- `validation_data=(X_val, y_val)` — evaluate on validation set after each epoch
- Adds `val_loss` and `val_accuracy` to the history object

## 7. Training with Callbacks: EarlyStopping

**File:** `6-train.py`

In [ ]:
def train_model_es(network, data, labels, batch_size, epochs,
                    validation_data=None, early_stopping=False,
                    patience=0, verbose=True, shuffle=False):
    callbacks = []
    if early_stopping and validation_data is not None:
        callbacks.append(
            K.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=patience
            )
        )
    return network.fit(
        x=data, y=labels,
        batch_size=batch_size, epochs=epochs,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=verbose, shuffle=shuffle
    )

**Key points:**
- `K.callbacks.EarlyStopping(monitor, patience)` — stop training when a monitored metric stops improving
- `monitor='val_loss'` — watch validation loss
- `patience` — number of epochs with no improvement before stopping

## 8. Training with Learning Rate Scheduler

**File:** `7-train.py`

In [ ]:
def train_model_lr(network, data, labels, batch_size,
                    epochs, validation_data=None, early_stopping=False,
                    patience=0, learning_rate_decay=False, alpha=0.1,
                    decay_rate=1, verbose=True, shuffle=False):
    callbacks = []
    if validation_data is not None:
        if early_stopping:
            callbacks.append(
                K.callbacks.EarlyStopping(monitor='val_loss', patience=patience)
            )
        if learning_rate_decay:
            def scheduler(epoch):
                return alpha / (1 + decay_rate * epoch)
            callbacks.append(
                K.callbacks.LearningRateScheduler(scheduler, verbose=1)
            )
    return network.fit(
        x=data, y=labels,
        batch_size=batch_size, epochs=epochs,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=verbose, shuffle=shuffle
    )

**Key points:**
- `K.callbacks.LearningRateScheduler(scheduler)` — adjust learning rate per epoch
- The `scheduler` function takes `epoch` and returns the new learning rate
- Common schedule: `alpha / (1 + decay_rate * epoch)` (time-based decay)

## 9. Training with ModelCheckpoint

**File:** `8-train.py`

In [ ]:
def train_model_mc(network, data, labels, batch_size,
                    epochs, validation_data=None, early_stopping=False,
                    patience=0, learning_rate_decay=False, alpha=0.1,
                    decay_rate=1, save_best=False, filepath=None,
                    verbose=True, shuffle=False):
    callbacks = []
    if validation_data is not None:
        if early_stopping:
            callbacks.append(
                K.callbacks.EarlyStopping(monitor='val_loss', patience=patience)
            )
        if learning_rate_decay:
            def scheduler(epoch):
                return alpha / (1 + decay_rate * epoch)
            callbacks.append(
                K.callbacks.LearningRateScheduler(scheduler, verbose=1)
            )
        if save_best:
            callbacks.append(
                K.callbacks.ModelCheckpoint(
                    filepath=filepath,
                    monitor='val_loss',
                    save_best_only=True,
                    mode='min'
                )
            )
    return network.fit(
        x=data, y=labels,
        batch_size=batch_size, epochs=epochs,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=verbose, shuffle=shuffle
    )

**Key points:**
- `K.callbacks.ModelCheckpoint(filepath, monitor, save_best_only, mode)` — save model weights at checkpoints
- `save_best_only=True` — only save when the monitored metric improves
- `mode='min'` — lower `val_loss` is better

## 10. Saving & Loading Models

**File:** `9-model.py`

In [ ]:
def save_model(network, filename):
    network.save(filename)

def load_model(filename):
    return K.models.load_model(filename)

**Key points:**
- `model.save(filename)` — saves architecture, weights, optimizer state, and config
- `K.models.load_model(filename)` — loads it back (no need to rebuild the model)

## 11. Saving & Loading Weights

**File:** `10-weights.py`

In [ ]:
def save_weights(network, filename, save_format='keras'):
    network.save_weights(filename, save_format=save_format)

def load_weights(network, filename):
    network.load_weights(filename)

**Key points:**
- `model.save_weights(filename)` — saves only the weights (not architecture)
- `model.load_weights(filename)` — loads weights into an existing model
- `save_format='keras'` — newer Keras weights format (vs `'h5'` for HDF5)

## 12. Saving & Loading Model Config (JSON)

**File:** `11-config.py`

In [ ]:
def save_config(network, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(network.to_json())

def load_config(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return K.models.model_from_json(f.read())

**Key points:**
- `model.to_json()` — serialize architecture to JSON string (weights not included)
- `K.models.model_from_json(json_string)` — reconstruct architecture from JSON (weights must be loaded separately)

## 13. Evaluating a Model

**File:** `12-test.py`

In [ ]:
def test_model(network, data, labels, verbose=True):
    return network.evaluate(
        x=data,
        y=labels,
        verbose=verbose
    )

**Key points:**
- `model.evaluate(x, y)` — compute loss and metrics on test data
- Returns a list: `[loss, accuracy, ...]` (whatever metrics were set in `compile`)

## 14. Making Predictions

**File:** `13-predict.py`

In [ ]:
def predict(network, data, verbose=False):
    return network.predict(data, verbose=verbose)

**Key points:**
- `model.predict(x)` — generate predictions for input samples
- Returns a numpy array of predicted probabilities (or values)

## Summary

| Topic | File | Key Function/Class |
|---|---|---|
| Sequential model | `0-sequential.py` | `K.Sequential()`, `.add()` |
| Functional API | `1-input.py` | `K.Input()`, `K.Model()` |
| Compile/optimize | `2-optimize.py` | `model.compile()`, `K.optimizers.Adam` |
| One-hot encoding | `3-one_hot.py` | `K.utils.to_categorical()` |
| Basic training | `4-train.py` | `model.fit()` |
| Validation data | `5-train.py` | `validation_data` param |
| EarlyStopping | `6-train.py` | `K.callbacks.EarlyStopping` |
| LR scheduler | `7-train.py` | `K.callbacks.LearningRateScheduler` |
| ModelCheckpoint | `8-train.py` | `K.callbacks.ModelCheckpoint` |
| Save/load model | `9-model.py` | `model.save()`, `K.models.load_model()` |
| Save/load weights | `10-weights.py` | `model.save_weights()`, `.load_weights()` |
| Save/load config | `11-config.py` | `model.to_json()`, `model_from_json()` |
| Evaluation | `12-test.py` | `model.evaluate()` |
| Prediction | `13-predict.py` | `model.predict()` |